# UrbanRisk AI — M1 (Multi-Zone): Chennai AQI Forecasting Across 20 Zones
Extends the single-point M1 notebook to a spatial grid. Same data sources, same honesty about what's
ground-truth vs reanalysis — see the M1 notebook intro for the full explanation. This notebook assumes
you've already run M1 successfully.

### Zones used here
20 zones centered on well-known Chennai localities. **Kodungaiyur, Royapuram, Perungudi, Koyambedu, and
Manali are real TNPCB CAAQM station locations** (verified against TNPCB's published station lists). The
remaining 15 are approximate locality centroids — good enough for a first spatial grid, but not surveyed
points, so refine with proper geocoding (e.g. Google Maps API or OSM Nominatim) before treating this as a
production zone boundary set.


## 1. Setup

In [ ]:
!pip install -q xgboost shap scikit-learn pandas numpy requests matplotlib

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (12, 4)

END_DATE = (datetime.utcnow() - timedelta(days=5)).strftime("%Y-%m-%d")
START_DATE = (datetime.utcnow() - timedelta(days=5) - timedelta(days=730)).strftime("%Y-%m-%d")  # ~2 years
print("Pulling data from", START_DATE, "to", END_DATE)


## 2. Define the zone grid
`is_caaqm_station=True` marks zones anchored on a real, verified TNPCB CAAQM station location.


In [ ]:
ZONES = [
    # zone_id,        name,                lat,      lon,      is_caaqm_station
    ("CHN_CENTRAL",   "Chennai Central",    13.0827,  80.2707,  False),
    ("CHN_MANALI",    "Manali",             13.1631,  80.2586,  True),
    ("CHN_KODUNGYR",  "Kodungaiyur",        13.1275,  80.2470,  True),
    ("CHN_ROYAPRM",   "Royapuram",          13.1143,  80.2947,  True),
    ("CHN_WSHRMNPT",  "Washermanpet",       13.1167,  80.2833,  False),
    ("CHN_PERAMBUR",  "Perambur",           13.1102,  80.2323,  False),
    ("CHN_KOYAMBDU",  "Koyambedu",          13.0694,  80.1948,  True),
    ("CHN_ANNANGR",   "Anna Nagar",         13.0850,  80.2101,  False),
    ("CHN_AMBATTUR",  "Ambattur",           13.1143,  80.1548,  False),
    ("CHN_AVADI",     "Avadi",              13.1147,  80.0970,  False),
    ("CHN_NUNGMBKM",  "Nungambakkam",       13.0569,  80.2425,  False),
    ("CHN_TNAGAR",    "T Nagar",            13.0418,  80.2341,  False),
    ("CHN_MYLAPORE",  "Mylapore",           13.0339,  80.2619,  False),
    ("CHN_ADYAR",     "Adyar",              13.0067,  80.2570,  False),
    ("CHN_GUINDY",    "Guindy",             13.0067,  80.2206,  False),
    ("CHN_VELACHRY",  "Velachery",          12.9756,  80.2200,  False),
    ("CHN_PERUNGDI",  "Perungudi",          12.9647,  80.2422,  True),
    ("CHN_SHOLINGL",  "Sholinganallur/OMR", 12.9010,  80.2279,  False),
    ("CHN_TAMBARAM",  "Tambaram",           12.9249,  80.1000,  False),
    ("CHN_PALLAVRM",  "Pallavaram",         12.9675,  80.1491,  False),
]

zones_df = pd.DataFrame(ZONES, columns=["zone_id", "zone_name", "latitude", "longitude", "is_caaqm_station"])
print(f"{len(zones_df)} zones defined ({zones_df['is_caaqm_station'].sum()} anchored on real CAAQM stations)")
zones_df


In [ ]:
fig, ax = plt.subplots(figsize=(6, 7))
ax.scatter(zones_df["longitude"], zones_df["latitude"], c=zones_df["is_caaqm_station"].map({True: "firebrick", False: "steelblue"}), s=60)
for _, row in zones_df.iterrows():
    ax.annotate(row["zone_name"], (row["longitude"], row["latitude"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_title("Chennai zone grid (red = verified CAAQM station location)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()


## 3. Fetch weather + air quality history per zone

In [ ]:
def fetch_weather_history(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": ",".join([
            "temperature_2m", "relative_humidity_2m", "pressure_msl",
            "precipitation", "rain", "wind_speed_10m", "wind_direction_10m",
        ]),
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df.rename(columns={
        "time": "timestamp", "temperature_2m": "temperature", "relative_humidity_2m": "humidity",
        "pressure_msl": "pressure", "precipitation": "rainfall_1h",
        "wind_speed_10m": "wind_speed", "wind_direction_10m": "wind_direction",
    })

def fetch_air_quality_history(lat, lon, start_date, end_date):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "pm2_5,pm10,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi",
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df.rename(columns={
        "time": "timestamp", "pm2_5": "PM2_5", "pm10": "PM10",
        "nitrogen_dioxide": "NO2", "sulphur_dioxide": "SO2", "ozone": "O3", "us_aqi": "AQI",
    })

all_zone_frames = []
for _, zone in zones_df.iterrows():
    print(f"Fetching {zone['zone_id']} ({zone['zone_name']})...")
    w = fetch_weather_history(zone["latitude"], zone["longitude"], START_DATE, END_DATE)
    time.sleep(0.5)  # be polite to the free API
    a = fetch_air_quality_history(zone["latitude"], zone["longitude"], START_DATE, END_DATE)
    time.sleep(0.5)

    merged = w.merge(a, on="timestamp", how="inner")
    merged["zone_id"] = zone["zone_id"]
    merged["zone_name"] = zone["zone_name"]
    merged["latitude"] = zone["latitude"]
    merged["longitude"] = zone["longitude"]
    merged["is_caaqm_station"] = zone["is_caaqm_station"]
    merged["aqi_source"] = "openmeteo_reanalysis"
    all_zone_frames.append(merged)

master = pd.concat(all_zone_frames, ignore_index=True)
print("\nMaster dataset shape:", master.shape)
print("Rows per zone:")
print(master.groupby("zone_id").size())


## 4. Inspect & clean (same logic as M1, applied per zone)

In [ ]:
weather_cols = ["temperature", "humidity", "pressure", "rainfall_1h", "rain", "wind_speed", "wind_direction"]
aq_cols = ["PM2_5", "PM10", "NO2", "SO2", "O3", "AQI"]

print("Missing values per column (whole dataset):")
print(master[weather_cols + aq_cols].isna().sum())

def clean_zone_group(g, cols, max_gap=3):
    g = g.sort_values("timestamp")
    g[cols] = g[cols].interpolate(method="linear", limit=max_gap, limit_direction="both")
    return g

master = master.groupby("zone_id", group_keys=False).apply(lambda g: clean_zone_group(g, weather_cols + aq_cols))
master = master.sort_values(["zone_id", "timestamp"]).reset_index(drop=True)

remaining_nans = master[weather_cols + aq_cols].isna().sum().sum()
print(f"\nRemaining NaNs after per-zone interpolation: {remaining_nans}")


## 5. Feature engineering — per zone (no cross-zone leakage)

In [ ]:
def add_features(g):
    g = g.sort_values("timestamp").reset_index(drop=True)

    g["hour"] = g["timestamp"].dt.hour
    g["day_of_week"] = g["timestamp"].dt.dayofweek
    g["month"] = g["timestamp"].dt.month
    g["is_weekend"] = (g["day_of_week"] >= 5).astype(int)

    lag_hours = [1, 3, 6, 12, 24]
    for col in ["AQI", "PM2_5", "PM10", "temperature", "humidity", "wind_speed"]:
        for lag in lag_hours:
            g[f"{col}_lag{lag}h"] = g[col].shift(lag)

    for col in ["AQI", "PM2_5", "temperature", "humidity"]:
        for w in [3, 6, 12, 24]:
            g[f"{col}_rollmean{w}h"] = g[col].shift(1).rolling(window=w, min_periods=max(2, w // 2)).mean()
            g[f"{col}_rollstd{w}h"] = g[col].shift(1).rolling(window=w, min_periods=max(2, w // 2)).std()

    for w in [1, 3, 6, 12, 24]:
        g[f"rainfall_{w}h"] = g["rainfall_1h"].shift(1).rolling(window=w, min_periods=1).sum()

    for h in [6, 12, 24]:
        g[f"AQI_target_{h}h"] = g["AQI"].shift(-h)

    return g

df = master.groupby("zone_id", group_keys=False).apply(add_features)
print("Shape after feature engineering:", df.shape)


## 6. Drop incomplete rows & prepare model matrix
Zone is kept as a categorical feature so one model learns shared patterns across all 20 zones while still telling them apart.

In [ ]:
feature_cols = [c for c in df.columns if any(k in c for k in ["lag", "roll", "rainfall_", "hour", "day_of_week", "month", "is_weekend"])]
target_cols = ["AQI_target_6h", "AQI_target_12h", "AQI_target_24h"]

model_df = df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
model_df["zone_id"] = model_df["zone_id"].astype("category")

# zone_id (categorical) + lat/lon let the model learn both zone identity and spatial position
model_feature_cols = feature_cols + ["zone_id", "latitude", "longitude"]

print(f"Rows usable for modeling: {len(model_df)} out of {len(df)}")
print(f"Zones represented: {model_df['zone_id'].nunique()}")


## 7. Chronological train / validation / test split
Split by **global timestamp**, not per zone — every zone's data in the test period is unseen at train time,
which is the realistic deployment scenario (predicting the future across the whole city at once).


In [ ]:
unique_ts = sorted(model_df["timestamp"].unique())
n = len(unique_ts)
train_cutoff = unique_ts[int(n * 0.70)]
val_cutoff = unique_ts[int(n * 0.85)]

train_df = model_df[model_df["timestamp"] < train_cutoff]
val_df = model_df[(model_df["timestamp"] >= train_cutoff) & (model_df["timestamp"] < val_cutoff)]
test_df = model_df[model_df["timestamp"] >= val_cutoff]

print(f"Train: {len(train_df)} rows  (< {train_cutoff})")
print(f"Val:   {len(val_df)} rows  ({train_cutoff} to {val_cutoff})")
print(f"Test:  {len(test_df)} rows  (>= {val_cutoff})")

X_train, X_val, X_test = train_df[model_feature_cols], val_df[model_feature_cols], test_df[model_feature_cols]


## 8. Baseline — persistence, evaluated per zone and overall

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    print(f"[{label}] MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}  MAPE={mape:.1f}%")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

results = {}
for h in [6, 12, 24]:
    y_true = test_df[f"AQI_target_{h}h"]
    y_pred_persistence = test_df["AQI"]
    results[f"persistence_{h}h"] = evaluate(y_true, y_pred_persistence, f"Persistence +{h}h (all zones)")


## 9. XGBoost — one multi-zone model per horizon

In [ ]:
from xgboost import XGBRegressor

models = {}
for h in [6, 12, 24]:
    y_train = train_df[f"AQI_target_{h}h"]
    y_val = val_df[f"AQI_target_{h}h"]

    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mae",
        early_stopping_rounds=30,
        enable_categorical=True,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    models[h] = model
    print(f"Trained XGBoost for +{h}h — best iteration: {model.best_iteration}")


## 10. Evaluation — overall and per-zone

In [ ]:
for h in [6, 12, 24]:
    y_true = test_df[f"AQI_target_{h}h"]
    y_pred = models[h].predict(X_test)
    results[f"xgboost_{h}h"] = evaluate(y_true, y_pred, f"XGBoost +{h}h (all zones)")

pd.DataFrame(results).T


In [ ]:
# Per-zone breakdown for the +6h model — useful for spotting zones the model struggles with
h = 6
test_eval = test_df.copy()
test_eval["y_pred"] = models[h].predict(X_test)
test_eval["y_true"] = test_eval[f"AQI_target_{h}h"]

per_zone = test_eval.groupby("zone_name").apply(
    lambda g: pd.Series({
        "MAE": mean_absolute_error(g["y_true"], g["y_pred"]),
        "RMSE": mean_squared_error(g["y_true"], g["y_pred"]) ** 0.5,
        "R2": r2_score(g["y_true"], g["y_pred"]),
        "n_rows": len(g),
    })
).sort_values("MAE")
per_zone


In [ ]:
per_zone["MAE"].plot(kind="barh", figsize=(8, 7), title="+6h AQI forecast — MAE by zone")
plt.gca().invert_yaxis()
plt.xlabel("MAE")
plt.tight_layout()
plt.show()


## 11. Actual vs predicted — one example zone

In [ ]:
example_zone = "Perungudi"  # change to inspect any zone
zone_test = test_eval[test_eval["zone_name"] == example_zone].sort_values("timestamp")

plt.figure(figsize=(14, 4))
plt.plot(zone_test["timestamp"], zone_test["y_true"], label="Actual", linewidth=0.8)
plt.plot(zone_test["timestamp"], zone_test["y_pred"], label="Predicted", linewidth=0.8, alpha=0.8)
plt.title(f"AQI +6h — Actual vs Predicted ({example_zone})")
plt.legend()
plt.tight_layout()
plt.show()


## 12. SHAP — does the model actually use zone identity and location?

In [ ]:
import shap

h = 6
# SHAP's TreeExplainer needs numeric categoricals; XGBoost's own category codes work fine here
X_test_shap = X_test.copy()
X_test_shap["zone_id"] = X_test_shap["zone_id"].cat.codes

explainer = shap.TreeExplainer(models[h])
shap_values = explainer(X_test_shap)

shap.summary_plot(shap_values, X_test_shap, max_display=15, show=True)


## 13. Save models, zone metadata, and per-zone metrics

In [ ]:
import joblib
import json
import os

os.makedirs("models", exist_ok=True)

for h in [6, 12, 24]:
    joblib.dump(models[h], f"models/aqi_xgboost_multizone_{h}h.joblib")

zones_df.to_csv("models/zones.csv", index=False)
per_zone.to_csv("models/per_zone_metrics_6h.csv")

metadata = {
    "feature_columns": model_feature_cols,
    "target_columns": target_cols,
    "n_zones": len(zones_df),
    "zone_ids": zones_df["zone_id"].tolist(),
    "train_range": [str(train_df['timestamp'].min()), str(train_df['timestamp'].max())],
    "val_range": [str(val_df['timestamp'].min()), str(val_df['timestamp'].max())],
    "test_range": [str(test_df['timestamp'].min()), str(test_df['timestamp'].max())],
    "aqi_data_source": "openmeteo_reanalysis (CAMS) — NOT raw CPCB ground readings",
    "weather_data_source": "openmeteo_archive (ERA5)",
    "test_metrics_overall": {k: v for k, v in results.items() if k.startswith("xgboost")},
}

with open("models/model_metadata_multizone.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved multi-zone models + metadata to ./models/")
